# Structured Data To Schema, Property Graph, And RDF

This notebook exercises the structured path: CSV/SQL-like rows → direct table mapping → schema proposal → SHACL/OWL-lite vocabulary → property graph and RDF output.

In [ ]:
from pathlib import Path
import sys, json, tempfile

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
from kg_construction.infrastructure.loaders.csv_loader import CsvLoader
from kg_construction.infrastructure.mapping.direct_table_mapper import (
    DirectTableMapper, DirectTableMappingConfig, ForeignKeyMappingConfig,
)
from kg_construction.infrastructure.ontology import SchemaProposalWorkflow, ShaclValidator
from kg_construction.infrastructure.ontology.validation import EvidenceRequiredValidator, SchemaRegistryValidator
from kg_construction.domain.schema import SchemaRegistry
from kg_construction.infrastructure.graph_builders import PropertyGraphBuilder, RDFGraphBuilder, EvidenceGraphBuilder
from kg_construction.infrastructure.graph_stores.exporters import RDFExporter

source = CsvLoader().load(ROOT / 'examples/data_to_kg/sample.csv')[0]
mapper = DirectTableMapper(DirectTableMappingConfig(
    table_name='people',
    node_type='Person',
    id_columns=['id'],
    name_column='name',
    foreign_keys=[ForeignKeyMappingConfig(
        source_column='manager_id', target_column='id', relationship_type='REPORTS_TO'
    )],
))
batch = mapper.map_source(source)
print({'entities': len(batch.entities), 'relations': len(batch.relations)})
print([(entity.entity_type, entity.name) for entity in batch.entities])
print([(relation.relation_type, relation.properties) for relation in batch.relations])

In [ ]:
workflow = SchemaProposalWorkflow()
proposal = workflow.propose([source], schema_id='sample_people')
schema = proposal.graph_schema
print(schema.status)
print([node.name for node in schema.node_types])
print([relationship.name for relationship in schema.relationship_types])
print('SHACL preview:')
print(chr(10).join(proposal.shacl_turtle.splitlines()[:12]))
print('OWL preview:')
print(chr(10).join((proposal.owl_turtle or '').splitlines()[:12]))

In [ ]:
approved = workflow.promote(workflow.approve(schema))
validation = SchemaRegistryValidator(SchemaRegistry(approved)).validate(batch)
evidence_validation = EvidenceRequiredValidator().validate(batch)
property_graph = PropertyGraphBuilder().build(batch, validation_report=evidence_validation)
rdf_graph = RDFGraphBuilder().build(batch, validation_report=evidence_validation)
evidence_graph = EvidenceGraphBuilder().build(batch)
shacl_result = ShaclValidator().validate_rdf_graph(rdf_graph, graph_schema=approved)
rdf_payload = RDFExporter().export_rdf_graph(rdf_graph)
print({
    'schema_registry_valid': validation.valid,
    'evidence_valid': evidence_validation.valid,
    'property_nodes': len(property_graph.nodes),
    'property_edges': len(property_graph.edges),
    'rdf_triples': len(rdf_graph.triples),
    'rdf_quads': len(rdf_graph.quads),
    'shacl_conforms': shacl_result.conforms,
    'named_graph_evidence': 'urn:kg:graph:evidence' in rdf_payload['nquads'],
})